## 1. Dependency

In [1]:
import csv
import re
import pandas as pd
from collections import defaultdict

## 2. Konfigurasi

In [ ]:
dataset = "1sample"

CSV_GROUND_TRUTH = f"results/{dataset}/result-3-low-level-ground-truth.csv"
CSV_LABEL_PREDICT = f"results/{dataset}/result-4-low-level-predict.csv"
CSV_OUTPUT = f"results/{dataset}/result-5-low-level-evaluation.csv"

## 3. Fungsi Helper

In [3]:
def count_lines(path):
    """Fast line count"""
    cnt = 0
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        for _ in f:
            cnt += 1
    return cnt

## 4. Fungsi untuk menghapus severity level

In [4]:
def remove_severity(value):
    """
    Menghapus severity level dari string.
    Contoh: 'Rule Name[high]' -> 'Rule Name'
    """
    if not value:
        return value
    # Hapus pattern [severity] di akhir string
    return re.sub(r'\[(critical|high|medium|low)\]$', '', value).strip()

## 5. Load dan Merge Data Berdasarkan event_id

In [5]:
print("Loading ground truth labels...")
ground_truth_dict = {}
gt_count = 0

with open(CSV_GROUND_TRUTH, 'r', encoding='utf-8', errors='replace') as f:
    reader = csv.DictReader(f)
    for row in reader:
        event_id = row.get('event_id', '')
        ground_truth_label = row.get('ground_truth_label', '')
        ground_truth_dict[event_id] = ground_truth_label
        gt_count += 1
        
        if gt_count % 100000 == 0:
            print(f"  Loaded {gt_count:,} ground truth labels...")

print(f"✓ Loaded {gt_count:,} ground truth labels")
print()

Loading ground truth labels...
  Loaded 100,000 ground truth labels...
  Loaded 200,000 ground truth labels...
✓ Loaded 216,754 ground truth labels



## 6. Evaluasi dan Export ke CSV

| Term | Definisi | Penjelasan |
|------|----------|------------|
| **True Positive (TP)** | `ground_truth_label` = attack **DAN** `label_predict` = attack | Ground truth adalah **attack**, Sigma **berhasil** mendeteksi sebagai attack |
| **True Negative (TN)** | `ground_truth_label` = benign **DAN** `label_predict` = benign | Ground truth adalah **benign**, Sigma **benar** mendeteksi sebagai benign |
| **False Positive (FP)** | `ground_truth_label` = benign **TAPI** `label_predict` = attack | Ground truth adalah **benign**, tapi Sigma **salah** mendeteksi sebagai attack |
| **False Negative (FN)** | `ground_truth_label` = attack **TAPI** `label_predict` = benign | Ground truth adalah **attack**, tapi Sigma **gagal** mendeteksi (dianggap benign) |

In [6]:
print("Starting evaluation and merge process...")
print()

total_lines = count_lines(CSV_LABEL_PREDICT)
print(f"Total lines in {CSV_LABEL_PREDICT}: {total_lines:,}")
print()

processed = 0
matched = 0
unmatched = 0

# Confusion Matrix counters
TP = 0 
TN = 0 
FP = 0 
FN = 0

with open(CSV_LABEL_PREDICT, 'r', encoding='utf-8', errors='replace') as f_predict, \
     open(CSV_OUTPUT, 'w', newline='', encoding='utf-8') as f_out:
    
    reader = csv.DictReader(f_predict)
    
    # Tambahkan kolom ground_truth_label dan status
    fieldnames = reader.fieldnames[:]
    
    # Cek apakah ground_truth_label sudah ada
    if 'ground_truth_label' not in fieldnames:
        # Insert ground_truth_label setelah decoded
        if 'decoded' in fieldnames:
            idx = fieldnames.index('decoded') + 1
            fieldnames.insert(idx, 'ground_truth_label')
        else:
            fieldnames.append('ground_truth_label')
    
    # Tambahkan status di akhir
    if 'status' not in fieldnames:
        fieldnames.append('status')
    
    writer = csv.DictWriter(f_out, fieldnames=fieldnames)
    writer.writeheader()
    
    for row in reader:
        processed += 1
        
        event_id = row.get('event_id', '')
        label_predict = row.get('label_predict', '')
        label_predict_clean = remove_severity(label_predict)
        
        # Cari ground_truth_label dari dictionary
        ground_truth_label = ground_truth_dict.get(event_id, '')
        
        if ground_truth_label:
            matched += 1
        else:
            unmatched += 1
        
        # Tambahkan ground_truth_label ke row
        row['ground_truth_label'] = ground_truth_label
        
        # Tentukan status
        label_is_benign = (ground_truth_label == 'benign')
        predict_is_benign = (label_predict_clean == 'benign')
        
        if ground_truth_label == '':  # Jika tidak ada ground truth
            status = 'UNKNOWN'
        elif label_is_benign and predict_is_benign:
            TN += 1
            status = 'TN'
        elif label_is_benign and not predict_is_benign:
            FP += 1
            status = 'FP'
        elif not label_is_benign and predict_is_benign:
            FN += 1
            status = 'FN'
        else:
            # keduanya bukan benign (attack)
            TP += 1
            status = 'TP'
        
        row['status'] = status
        
        writer.writerow(row)
        
        # Progress tiap 50.000 baris
        if processed % 50000 == 0:
            print(f"Processed {processed:,}/{total_lines:,} lines ({processed/total_lines:.2%})")

print(f"\nEvaluation finished: {processed:,}/{total_lines:,} lines processed.")
print(f"Matched events: {matched:,}")
print(f"Unmatched events: {unmatched:,}")
print()
print(f"Output saved to: {CSV_OUTPUT}")

Starting evaluation and merge process...

Total lines in results/organization-x/result-4-low-level-predict.csv: 216,761

Processed 50,000/216,761 lines (23.07%)
Processed 100,000/216,761 lines (46.13%)
Processed 150,000/216,761 lines (69.20%)
Processed 200,000/216,761 lines (92.27%)

Evaluation finished: 216,754/216,761 lines processed.
Matched events: 216,754
Unmatched events: 0

Output saved to: results/organization-x/result-5-low-level-evaluation.csv


## 7. Tampilkan Summary dan Metrics

In [7]:
print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Total rows processed: {processed:,}")
print(f"Matched events:       {matched:,}")
print(f"Unmatched events:     {unmatched:,}")
print()
print(f"  True Positive  (TP): {TP}")
print(f"  True Negative  (TN): {TN}")
print(f"  False Positive (FP): {FP}")
print(f"  False Negative (FN): {FN}")
print()

# Metrics (hanya untuk data yang matched)
total_evaluated = TP + TN + FP + FN

if total_evaluated > 0:
    accuracy = (TP + TN) / total_evaluated * 100
    precision = TP / (TP + FP) * 100 if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) * 100 if (TP + FN) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print("-" * 70)
    print("METRICS (Based on matched events only)")
    print("-" * 70)
    print(f"  Evaluated events: {total_evaluated:,}")
    print()
    print(f"  Accuracy:  {accuracy:.2f}%  (TP + TN) / Total")
    print(f"  Precision: {precision:.2f}%  TP / (TP + FP)")
    print(f"  Recall:    {recall:.2f}%  TP / (TP + FN)")
    print(f"  F1-Score:  {f1_score:.2f}%  2 * (Precision * Recall) / (Precision + Recall)")
else:
    print("⚠ No events could be evaluated (no matched ground truth labels)")

print("=" * 70)

SUMMARY
Total rows processed: 216,754
Matched events:       216,754
Unmatched events:     0

  True Positive  (TP): 624
  True Negative  (TN): 215943
  False Positive (FP): 139
  False Negative (FN): 48

----------------------------------------------------------------------
METRICS (Based on matched events only)
----------------------------------------------------------------------
  Evaluated events: 216,754

  Accuracy:  99.91%  (TP + TN) / Total
  Precision: 81.78%  TP / (TP + FP)
  Recall:    92.86%  TP / (TP + FN)
  F1-Score:  86.97%  2 * (Precision * Recall) / (Precision + Recall)


## 8. Optional: Analisis Lebih Detail

In [8]:
# Load hasil untuk analisis lebih lanjut jika diperlukan
# df = pd.read_csv(CSV_OUTPUT)
# print(df['status'].value_counts())
# print()
# print(df[df['status'] == 'FP'][['event_id', 'ground_truth_label', 'label_predict']].head(10))